# Build the reporting views

Generated from `fabric/05-gold.yaml` — do not edit by hand.

Runs **after every dimension and fact**, because these views select from tables that Spark creates through the warehouse connector. They cannot exist before the first load, which is why they are built here rather than by a deploy-time migration.

`CREATE OR ALTER`, so a re-run converges instead of failing.

In [ ]:
WAREHOUSE = 'wh_gold'
REPORTING_SCHEMA = 'bi'
PHYSICAL_SCHEMA = 'dbo'

VIEWS = ['vw_daily_occupancy', 'vw_hourly_arrival_pattern', 'vw_company_presence']
STATEMENTS = ['CREATE OR ALTER VIEW [bi].[vw_daily_occupancy] AS\n'
 'SELECT d.full_date, d.day_name, d.is_work_day, b.building_name,\n'
 '       COUNT_BIG(*) AS arrival_count,\n'
 '       COUNT(DISTINCT f.card_holder_sk) AS distinct_people,\n'
 '       SUM(f.personnel_arrivals) AS personnel_arrivals,\n'
 '       SUM(f.visitor_arrivals) AS visitor_arrivals\n'
 'FROM dbo.fct_arrivals f JOIN dbo.dim_date d ON d.date_sk = f.arrival_date_sk JOIN '
 'dbo.dim_building b ON b.building_sk = f.building_sk GROUP BY d.full_date, d.day_name, '
 'd.is_work_day, b.building_name',
 'CREATE OR ALTER VIEW [bi].[vw_hourly_arrival_pattern] AS\n'
 'SELECT b.building_name, t.hour_of_day,\n'
 '       COUNT_BIG(*) AS arrival_count,\n'
 '       COUNT(DISTINCT f.card_holder_sk) AS distinct_people\n'
 'FROM dbo.fct_arrivals f JOIN dbo.dim_time t ON t.time_sk = f.arrival_time_sk JOIN '
 'dbo.dim_building b ON b.building_sk = f.building_sk GROUP BY b.building_name, t.hour_of_day',
 'CREATE OR ALTER VIEW [bi].[vw_company_presence] AS\n'
 'SELECT f.company_name,\n'
 '       DATEFROMPARTS(YEAR(d.full_date), MONTH(d.full_date), 1) AS month_start,\n'
 '       COUNT_BIG(*) AS arrival_count,\n'
 '       COUNT(DISTINCT f.card_holder_sk) AS distinct_people\n'
 'FROM dbo.fct_arrivals f JOIN dbo.dim_date d ON d.date_sk = f.arrival_date_sk WHERE '
 'f.company_name IS NOT NULL GROUP BY f.company_name, DATEFROMPARTS(YEAR(d.full_date), '
 'MONTH(d.full_date), 1)']


In [ ]:
import com.microsoft.spark.fabric  # noqa: F401  registers the connector
import requests

workspace_id = spark.conf.get('trident.workspace.id')
token = mssparkutils.credentials.getToken('https://api.fabric.microsoft.com')

# Resolved at RUN time from the workspace this notebook is in, so the
# same artefact points at dev's warehouse in dev and qa's in qa.
warehouses = requests.get(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses',
    headers={'Authorization': f'Bearer {token}'}, timeout=60).json()['value']
target = next(w for w in warehouses if w['displayName'] == WAREHOUSE)
endpoint = target['properties']['connectionString']
print(f'{WAREHOUSE} -> {endpoint}')


In [ ]:
sql_token = mssparkutils.credentials.getToken('https://database.windows.net/')
jvm = spark._jvm
props = jvm.java.util.Properties()
props.setProperty('accessToken', sql_token)
props.setProperty('encrypt', 'true')
conn = jvm.java.sql.DriverManager.getConnection(
    f'jdbc:sqlserver://{endpoint}:1433;database={WAREHOUSE}', props)
conn.setAutoCommit(True)
stmt = conn.createStatement()

# Each view is applied independently. One malformed view must not stop
# the other three -- and a single aggregate error hides how many were
# actually fine.
failed = []
for name, sql in zip(VIEWS, STATEMENTS):
    try:
        stmt.execute(sql)
        print(f'  ok      {REPORTING_SCHEMA}.{name}')
    except Exception as exc:
        failed.append(name)
        print(f'  FAILED  {REPORTING_SCHEMA}.{name}: {str(exc)[:160]}')

stmt.close(); conn.close()

if failed:
    raise RuntimeError(
        f'{len(failed)} of {len(VIEWS)} view(s) could not be created: '
        + ', '.join(failed))
print(f'\nall {len(VIEWS)} reporting view(s) are current')
